In [2]:
# =========================
# 配置（按需修改）
# =========================
DATA_PATH = "/content/drive/MyDrive/demo_200000.csv"   # 支持 CSV 或 Excel
SHEET_NAME = None                       # Excel 时可指定；CSV 忽略

# 特征版本：'no_region'（不带地区）或 'region_onehot'（地区独热）
VERSION = "no_region"
# 若用 'region_onehot'，把地区列名写这里（可多个）
REGION_COLS = ["UK.Biobank.assessment.centre...Instance.0"]

# 训练超参
TEST_SIZE = 0.2
EPOCHS = 1
BATCH_SIZE = 256
LR = 1e-3
SEED = 42

# TabM / Backbone 配置
ARCH_TYPE = "tabm"   # 'plain' | 'tabm' | 'tabm-mini' | 'tabm-packed'
TABM_K = 32          # mini-ensemble 宽度（tabm/tabm-packed 生效）
BACKBONE_CFG = dict(n_blocks=3, d_hidden=512, dropout=0.2)

# 可选：保存模型与预处理器（为空不保存）
SAVE_DIR = ""   # 例如 "/mnt/data/tabm_artifacts"


In [3]:
!pip -q install paddlepaddle -i https://pypi.tuna.tsinghua.edu.cn/simple

# 验证
import paddle
paddle.utils.run_check()
print("Paddle version:", paddle.__version__)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 189.0/189.0 MB 6.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 6.5 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:718: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)


Running verify PaddlePaddle program ... 
PaddlePaddle works well on 1 CPU.
PaddlePaddle is installed successfully! Let's start deep learning with PaddlePaddle now.
Paddle version: 3.2.0


/usr/local/lib/python3.12/dist-packages/paddle/pir/math_op_patch.py:219: UserWarning: Value do not have 'place' interface for pir graph mode, try not to use it. None will be returned.
  warnings.warn(


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
# =========================
# 依赖导入
# =========================
import os, json, math, warnings
warnings.filterwarnings("ignore", category=UserWarning)

import numpy as np
import pandas as pd
from typing import List, Dict, Any, Tuple, Literal, Optional

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score, precision_score, recall_score,
    hamming_loss, average_precision_score, multilabel_confusion_matrix
)
from tqdm.auto import tqdm

import paddle
import paddle.nn as nn
import paddle.nn.functional as F
from paddle.io import Dataset, DataLoader

def seed_everything(seed: int = 42):
    import random, os
    random.seed(seed); np.random.seed(seed)
    paddle.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

seed_everything(SEED)
paddle.set_device("cpu")   # 强制使用 CPU


Place(cpu)

In [6]:
# =========================
# 读入数据（CSV / Excel 自适应）
# =========================
if DATA_PATH.lower().endswith(".csv"):
    df = pd.read_csv(DATA_PATH)
else:
    df = pd.read_excel(DATA_PATH, sheet_name=SHEET_NAME) if SHEET_NAME else pd.read_excel(DATA_PATH)

df.shape, df.head(3)


((200000, 434),
         ID UK.Biobank.assessment.centre...Instance.0  cataract cataract_time  \
 0  3203000                                     Leeds         1    2011-09-28   
 1  4084800                         Stockport (pilot)         0           NaN   
 2  1759102                                   Glasgow         0           NaN   
 
    glaucoma glaucoma_time  AMD AMD_time  DR DR_time  ... Operation_Code  \
 0         0           NaN    0      NaN   0     NaN  ...              1   
 1         0           NaN    0      NaN   0     NaN  ...              1   
 2         0           NaN    0      NaN   0     NaN  ...              1   
 
    Home_Area_Population_Density  Pulse_Rate      FEV1         PEF  \
 0                           0.0        66.0  1.876667  197.333333   
 1                           0.0        65.5       NaN         NaN   
 2                           1.0         NaN  3.966667  442.000000   
 
    Incorrect_Matches  Diastolic_BP  Systolic_BP   FVC  Total_Bilirubi

In [7]:
# =========================
# 工具 & 常量
# =========================
EXCLUDE_COLS = ["ID","DR","DR_time","AMD_time","AMD","glaucoma_time","glaucoma","cataract_time","cataract"]
LABEL_COLS   = ["DR","AMD","glaucoma","cataract"]

def make_onehot_encoder():
    # 兼容不同 sklearn 版本
    try:
        return OneHotEncoder(sparse_output=False, handle_unknown="ignore")
    except TypeError:
        return OneHotEncoder(sparse=False, handle_unknown="ignore")

def safe_auc(y_true: np.ndarray, y_prob: np.ndarray, average: str) -> float:
    try: return roc_auc_score(y_true, y_prob, average=average)
    except Exception: return float("nan")

def safe_ap(y_true: np.ndarray, y_prob: np.ndarray, average: str) -> float:
    try: return average_precision_score(y_true, y_prob, average=average)
    except Exception: return float("nan")

def per_label_auc_ap(y_true: np.ndarray, y_prob: np.ndarray, label_names: List[str]) -> Dict[str, Dict[str, float]]:
    out = {}
    for i, name in enumerate(label_names):
        yt, yp = y_true[:, i], y_prob[:, i]
        if len(np.unique(yt)) >= 2:
            out[name] = {
                "roc_auc": roc_auc_score(yt, yp),
                "ap":      average_precision_score(yt, yp)
            }
        else:
            out[name] = {"roc_auc": float("nan"), "ap": float("nan")}
    return out

def per_label_confusion(y_true: np.ndarray, y_pred: np.ndarray, label_names: List[str]) -> Dict[str, Dict[str, int]]:
    cms = multilabel_confusion_matrix(y_true, y_pred)
    out = {}
    for i, name in enumerate(label_names):
        tn, fp, fn, tp = cms[i].ravel().tolist()
        out[name] = {"tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)}
    return out


In [8]:
# =========================
# 特征构建（两版：不带地区 / 地区独热）
# - 数值/布尔列：均值填充 + 标准化
# - 仅当 region_onehot 时，对 REGION_COLS 做 OneHot
# - 其它非数值列丢弃
# =========================
def build_features_labels(
    df: pd.DataFrame,
    version: str,
    region_cols: List[str]
) -> Tuple[np.ndarray, np.ndarray, Dict[str, Any]]:
    assert all(c in df.columns for c in LABEL_COLS), f"缺少标签列：{LABEL_COLS}"
    y = df[LABEL_COLS].astype(float).values

    base_drop = list(set(EXCLUDE_COLS))
    feat_df = df.drop(columns=[c for c in base_drop if c in df.columns], errors="ignore")

    existing_regions = []
    if version == "region_onehot" and region_cols:
        existing_regions = [c for c in region_cols if c in feat_df.columns]
        feat_df = feat_df.drop(columns=existing_regions, errors="ignore")

    # 尝试数值化（失败置 NaN）
    X_num_try = feat_df.apply(pd.to_numeric, errors="coerce")
    keep_cols = [c for c in X_num_try.columns if not X_num_try[c].isna().all()]
    if len(keep_cols) == 0:
        raise ValueError("没有可用的数值特征列。请检查数据，或设置 VERSION='region_onehot' 并正确指定 REGION_COLS。")
    X_num = X_num_try[keep_cols].copy()
    for c in X_num.columns:
        if X_num[c].dtype == bool:
            X_num[c] = X_num[c].astype(float)

    # 数值列：均值填充 + 标准化
    imputer = SimpleImputer(strategy="mean")
    scaler  = StandardScaler()
    X_num_imp = imputer.fit_transform(X_num.values)
    X_num_std = scaler.fit_transform(X_num_imp)

    meta: Dict[str, Any] = {
        "version": version,
        "kept_numeric_cols": keep_cols,
        "used_region_cols": existing_regions,
        "imputer": imputer,
        "scaler":  scaler,
        "enc": None,
        "feature_dim_before_region": X_num_std.shape[1]
    }

    if version == "region_onehot" and existing_regions:
        enc = make_onehot_encoder()
        X_region = enc.fit_transform(df[existing_regions].fillna("missing"))
        X = np.hstack([X_num_std, X_region])
        meta["enc"] = enc
        meta["region_onehot_dim"] = X_region.shape[1]
    else:
        X = X_num_std
        meta["region_onehot_dim"] = 0

    meta["final_input_dim"] = X.shape[1]
    return X, y, meta


In [9]:
# =========================
# TabM 组件（Paddle 实现）
# =========================
def init_rsqrt_uniform_(w: paddle.Tensor) -> paddle.Tensor:
    bound = 1.0 / math.sqrt(w.shape[-1])
    noise = paddle.uniform(w.shape, min=-bound, max=bound, dtype=w.dtype)
    w.set_value(noise); return w

def init_random_signs_(w: paddle.Tensor) -> paddle.Tensor:
    with paddle.no_grad():
        p = paddle.full(w.shape, 0.5, dtype='float32')
        s = paddle.bernoulli(p) * 2.0 - 1.0
        s = paddle.cast(s, w.dtype)
        w.set_value(s)
    return w

class NLinear(nn.Layer):
    """PackedEnsemble: K 份 Linear 打包 → 输入 (B,K,D), 权重 (K, I, O)"""
    def __init__(self, k: int, in_f: int, out_f: int, bias: bool = True):
        super().__init__()
        self.k, self.in_f, self.out_f = k, in_f, out_f
        self.weight = self.create_parameter(shape=[k, in_f, out_f])
        self.bias_e = self.create_parameter(shape=[k, out_f]) if bias else None
        self.reset_parameters()

    def reset_parameters(self):
        init_rsqrt_uniform_(self.weight)
        if self.bias_e is not None:
            init_rsqrt_uniform_(self.bias_e)

    def forward(self, x):              # x: (B,K,I)
        xk = paddle.transpose(x, [1, 0, 2])      # (K,B,I)
        yk = paddle.bmm(xk, self.weight)         # (K,B,O)
        y  = paddle.transpose(yk, [1, 0, 2])     # (B,K,O)
        if self.bias_e is not None:
            y = y + self.bias_e
        return y

class ScaleEnsemble(nn.Layer):
    def __init__(self, k: int, d: int, init='ones'):
        super().__init__()
        self.k, self.d = k, d
        self.weight = self.create_parameter(shape=[k, d])
        self.init = init; self.reset_parameters()
    def reset_parameters(self):
        if self.init == 'ones':
            self.weight.set_value(paddle.ones_like(self.weight))
        else:
            init_random_signs_(self.weight)
    def forward(self, x):              # (B,K,D)
        return x * self.weight

class LinearBE(nn.Layer):
    """BatchEnsemble Linear:
       y_e = ((x * r_e) @ W) * s_e + b_e
       x: (B,K,I) → y: (B,K,O)
    """
    def __init__(self, in_f: int, out_f: int, k: int, scale_init='ones', bias: bool = True):
        super().__init__()
        self.k, self.in_f, self.out_f = k, in_f, out_f
        self.weight = self.create_parameter(shape=[in_f, out_f])   # 共享权重
        self.r = self.create_parameter(shape=[k, in_f])
        self.s = self.create_parameter(shape=[k, out_f])
        self.use_bias = bias
        self.bias_e = self.create_parameter(shape=[k, out_f]) if bias else None
        self.scale_init = scale_init
        self.reset_parameters()

    def reset_parameters(self):
        init_rsqrt_uniform_(self.weight)
        if self.scale_init == 'ones':
            self.r.set_value(paddle.ones_like(self.r))
            self.s.set_value(paddle.ones_like(self.s))
        else:
            init_random_signs_(self.r); init_random_signs_(self.s)
        if self.use_bias:
            init_rsqrt_uniform_(self.bias_e)

    def forward(self, x):              # (B,K,I)
        xr = x * self.r                                # (B,K,I)
        y  = paddle.matmul(xr, self.weight)            # (B,K,O)
        y  = y * self.s
        if self.use_bias:
            y = y + self.bias_e
        return y

class MLPBlock(nn.Layer):
    def __init__(self, d_in, d_hid, dropout, act='ReLU'):
        super().__init__()
        Act = getattr(nn, act)
        self.net = nn.Sequential(
            nn.Linear(d_in, d_hid),
            Act(),
            nn.Dropout(dropout),
        )
    def forward(self, x): return self.net(x)

class BackboneMLP(nn.Layer):
    def __init__(self, n_blocks: int, d_in: int, d_hidden: int, dropout: float):
        super().__init__()
        blocks = []
        for i in range(n_blocks):
            blocks.append(MLPBlock(d_in if i==0 else d_hidden, d_hidden, dropout))
        self.blocks = nn.LayerList(blocks)
    def forward(self, x):
        for blk in self.blocks:
            x = blk(x)
        return x

def _get_parent_by_path(root: nn.Layer, path_list):
    cur = root
    for p in path_list:
        if hasattr(cur, p):
            cur = getattr(cur, p)
        else:
            sub_layers = getattr(cur, "_sub_layers", None)
            if sub_layers is None or p not in sub_layers:
                raise AttributeError(f"Cannot locate sublayer '{p}' under '{type(cur).__name__}'")
            cur = sub_layers[p]
    return cur

def _replace_linear(module: nn.Layer, k: int, mode: Literal['be','packed']):
    to_replace = []
    for full_name, layer in module.named_sublayers(include_self=False):
        if isinstance(layer, nn.Linear):
            parts = full_name.split('.')
            parent_path, child_name = parts[:-1], parts[-1]
            parent = _get_parent_by_path(module, parent_path) if parent_path else module
            in_f  = layer.weight.shape[0]
            out_f = layer.weight.shape[1]
            if mode == 'be':
                new_layer = LinearBE(in_f, out_f, k)
                with paddle.no_grad():
                    new_layer.weight.set_value(layer.weight.clone())
                    if layer.bias is not None and new_layer.bias_e is not None:
                        b = layer.bias.reshape([1,-1]).tile([k,1])
                        new_layer.bias_e.set_value(b)
            else:  # packed
                new_layer = NLinear(k, in_f, out_f, bias=layer.bias is not None)
                with paddle.no_grad():
                    w = layer.weight.unsqueeze(0).tile([k,1,1])
                    new_layer.weight.set_value(w)
                    if layer.bias is not None and new_layer.bias_e is not None:
                        b = layer.bias.unsqueeze(0).tile([k,1])
                        new_layer.bias_e.set_value(b)
            to_replace.append((parent, child_name, new_layer))
    for parent, child_name, new_layer in to_replace:
        if hasattr(parent, child_name):
            setattr(parent, child_name, new_layer)
        else:
            parent._sub_layers[child_name] = new_layer

class TabMFeatureExtractor(nn.Layer):
    """arch_type: 'plain' | 'tabm' | 'tabm-mini' | 'tabm-packed'"""
    def __init__(self,
                 num_features: int,
                 arch_type: Literal['plain','tabm','tabm-mini','tabm-packed']='tabm',
                 k: int = 32,
                 backbone_cfg: Optional[dict] = None,
                 reduce: bool = True):
        super().__init__()
        if arch_type == 'plain':
            k = 1
        self.k = k
        self.reduce = reduce
        cfg = backbone_cfg or dict(n_blocks=3, d_hidden=512, dropout=0.1)
        self.d_hidden = cfg["d_hidden"]
        self.backbone = BackboneMLP(**cfg, d_in=num_features)

        if arch_type == 'tabm':
            _replace_linear(self.backbone, k, mode='be')
            self.min_adapter = None
        elif arch_type == 'tabm-mini':
            self.min_adapter = ScaleEnsemble(k, num_features, init='random-signs')
        elif arch_type == 'tabm-packed':
            _replace_linear(self.backbone, k, mode='packed')
            self.min_adapter = None
        else:
            self.min_adapter = None

    def forward(self, x_num: paddle.Tensor):      # x_num: (B, D)
        if self.k > 1:
            x = x_num.unsqueeze(1).tile([1, self.k, 1])  # (B,K,D)
        else:
            x = x_num.unsqueeze(1)                        # (B,1,D)
        if self.min_adapter is not None:
            x = self.min_adapter(x)
        feats = self.backbone(x)                          # (B,K,H)
        return feats.mean(axis=1) if self.reduce else feats  # (B,H) 或 (B,K,H)


In [10]:
# =========================
# 数据集 & 评估 & 训练（Paddle）
# =========================
class NumpyDataset(Dataset):
    def __init__(self, X: np.ndarray, y: np.ndarray):
        self.X = X.astype("float32")
        self.y = y.astype("float32")
    def __len__(self): return self.X.shape[0]
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

class TabMClassifier(nn.Layer):
    def __init__(self, input_dim: int, arch_type: str, k: int, backbone_cfg: dict, out_dim: int = 4):
        super().__init__()
        self.feat = TabMFeatureExtractor(
            num_features=input_dim,
            arch_type=arch_type,
            k=k,
            backbone_cfg=backbone_cfg,
            reduce=True
        )
        self.head = nn.Linear(self.feat.d_hidden, out_dim)  # 输出logits

    def forward(self, x):      # x: (B, D)
        h = self.feat(x)       # (B, H)
        logits = self.head(h)  # (B, 4)
        return logits

@paddle.no_grad()
def evaluate(model: nn.Layer, Xv: np.ndarray, yv: np.ndarray, thr: float = 0.5) -> Dict[str, Any]:
    model.eval()
    Xv_t = paddle.to_tensor(Xv.astype("float32"))
    logits = model(Xv_t)
    probs = F.sigmoid(logits).numpy()
    y_true = yv.astype(int)
    y_pred = (probs >= thr).astype(int)

    metrics = {
        "subset_accuracy": accuracy_score(y_true, y_pred),
        "hamming_loss": hamming_loss(y_true, y_pred)
    }
    for avg in ["micro", "macro", "weighted"]:
        metrics[f"precision_{avg}"] = precision_score(y_true, y_pred, average=avg, zero_division=0)
        metrics[f"recall_{avg}"]    = recall_score(y_true, y_pred, average=avg, zero_division=0)
        metrics[f"f1_{avg}"]        = f1_score(y_true, y_pred, average=avg, zero_division=0)
    metrics["roc_auc_macro"] = safe_auc(y_true, probs, average="macro")
    metrics["roc_auc_micro"] = safe_auc(y_true, probs, average="micro")
    metrics["pr_auc_macro"]  = safe_ap(y_true, probs, average="macro")
    metrics["pr_auc_micro"]  = safe_ap(y_true, probs, average="micro")
    metrics["per_label_auc_ap"]    = per_label_auc_ap(y_true, probs, LABEL_COLS)
    metrics["per_label_confusion"] = per_label_confusion(y_true, y_pred, LABEL_COLS)
    return metrics

from tqdm.auto import tqdm
import paddle
import paddle.nn as nn
import paddle.nn.functional as F

def train(model: nn.Layer,
          train_loader,
          X_val: np.ndarray,
          y_val: np.ndarray,
          epochs: int = 20,
          lr: float = 1e-3):
    model.train()
    opt = paddle.optimizer.Adam(learning_rate=lr, parameters=model.parameters())
    history = []

    for ep in range(1, epochs + 1):
        model.train()
        running, seen = 0.0, 0
        pbar = tqdm(train_loader, desc=f"Epoch {ep}/{epochs}", leave=False)

        for xb, yb in pbar:
            # forward
            logits = model(xb)  # (B,4)
            loss = F.binary_cross_entropy_with_logits(logits, yb, reduction='mean')

            # backward
            loss.backward()
            opt.step()
            opt.clear_grad()

            # 记录损失 —— 关键修复：使用 loss.item()（或 float(loss.numpy())）
            bs = xb.shape[0]
            running += loss.item() * bs
            seen    += bs
            pbar.set_postfix(loss=f"{running/max(seen,1):.4f}")

        tr_loss = running / max(seen, 1)

        # 验证
        ev = evaluate(model, X_val, y_val)
        ev["epoch"] = ep
        ev["train_loss"] = tr_loss
        history.append(ev)

        print(f"[Epoch {ep:03d}] loss={tr_loss:.4f}  "
              f"F1(micro)={ev['f1_micro']:.4f}  "
              f"AUC(macro)={ev['roc_auc_macro']:.4f}  "
              f"PR-AUC(macro)={ev['pr_auc_macro']:.4f}")

    return model, history



In [11]:
# =========================
# 构建特征 & 划分数据集 & 训练
# =========================
X, y, meta = build_features_labels(df, VERSION, REGION_COLS)
X_tr, X_va, y_tr, y_va = train_test_split(X, y, test_size=TEST_SIZE, random_state=SEED)

train_ds = NumpyDataset(X_tr, y_tr)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=False)

model = TabMClassifier(
    input_dim=X_tr.shape[1],
    arch_type=ARCH_TYPE,
    k=TABM_K,
    backbone_cfg=BACKBONE_CFG,
    out_dim=len(LABEL_COLS)
)

model, history = train(model, train_loader, X_va, y_va, epochs=EPOCHS, lr=LR)

final_eval = evaluate(model, X_va, y_va)
pd.DataFrame([{
    "subset_accuracy": final_eval["subset_accuracy"],
    "hamming_loss": final_eval["hamming_loss"],
    "precision_micro": final_eval["precision_micro"],
    "recall_micro": final_eval["recall_micro"],
    "f1_micro": final_eval["f1_micro"],
    "precision_macro": final_eval["precision_macro"],
    "recall_macro": final_eval["recall_macro"],
    "f1_macro": final_eval["f1_macro"],
    "roc_auc_macro": final_eval["roc_auc_macro"],
    "roc_auc_micro": final_eval["roc_auc_micro"],
    "pr_auc_macro": final_eval["pr_auc_macro"],
    "pr_auc_micro": final_eval["pr_auc_micro"],
}])


Epoch 1/1:   0%|          | 0/625 [00:00<?, ?it/s]

[Epoch 001] loss=0.1582  F1(micro)=0.1219  AUC(macro)=0.7941  PR-AUC(macro)=0.1947


,subset_accuracy,hamming_loss,precision_micro,recall_micro,f1_micro,precision_macro,recall_macro,f1_macro,roc_auc_macro,roc_auc_micro,pr_auc_macro,pr_auc_micro
0,0.831475,0.052219,0.455975,0.070363,0.121913,0.240722,0.050557,0.083563,0.794074,0.859987,0.194672,0.27163


In [12]:
# =========================
# 可选：保存模型与预处理器
# =========================
def save_artifacts(save_dir: str, model: nn.Layer, meta: Dict[str, Any]):
    os.makedirs(save_dir, exist_ok=True)
    paddle.save(model.state_dict(), os.path.join(save_dir, "model.pdparams"))
    try:
        import joblib
        joblib.dump(meta["imputer"], os.path.join(save_dir, "imputer.joblib"))
        joblib.dump(meta["scaler"],  os.path.join(save_dir, "scaler.joblib"))
        if meta.get("enc", None) is not None:
            joblib.dump(meta["enc"], os.path.join(save_dir, "onehot_encoder.joblib"))
        with open(os.path.join(save_dir, "meta.json"), "w", encoding="utf-8") as f:
            json.dump({
                "LABEL_COLS": LABEL_COLS,
                "EXCLUDE_COLS": EXCLUDE_COLS,
                "used_region_cols": meta.get("used_region_cols", []),
                "feature_dim_before_region": meta.get("feature_dim_before_region", None),
                "region_onehot_dim": meta.get("region_onehot_dim", None),
                "final_input_dim": meta.get("final_input_dim", None),
                "version": meta.get("version", None),
                "arch_type": ARCH_TYPE,
                "k": TABM_K,
                "backbone_cfg": BACKBONE_CFG,
            }, f, ensure_ascii=False, indent=2)
        print(f"[INFO] Artifacts saved to: {save_dir}")
    except Exception as e:
        print(f"[WARN] Failed to save preprocessors: {e}")

if SAVE_DIR:
    save_artifacts(SAVE_DIR, model, meta)


In [ ]:
# =========================
# （可选）快速对比 no_region vs region_onehot（若地区列存在）
# =========================
def run_variant(version: str, region_cols: List[str], epochs= max(1, EPOCHS//2)):
    Xv, yv, metav = build_features_labels(df, version, region_cols)
    X_tr, X_va, y_tr, y_va = train_test_split(Xv, yv, test_size=TEST_SIZE, random_state=SEED)
    ds = NumpyDataset(X_tr, y_tr)
    loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
    m = TabMClassifier(
        input_dim=X_tr.shape[1],
        arch_type=ARCH_TYPE,
        k=TABM_K,
        backbone_cfg=BACKBONE_CFG,
        out_dim=len(LABEL_COLS)
    )
    m, _ = train(m, loader, X_va, y_va, epochs=epochs, lr=LR)
    ev = evaluate(m, X_va, y_va)
    return ev

try:
    ev_no_region = run_variant("no_region", REGION_COLS)
    existing_regions = [c for c in REGION_COLS if c in df.columns]
    if existing_regions:
        ev_region = run_variant("region_onehot", REGION_COLS)
        comp = pd.DataFrame([
            {"version": "no_region", **{k: ev_no_region[k] for k in [
                "subset_accuracy","hamming_loss","f1_micro","f1_macro","roc_auc_macro","pr_auc_macro"
            ]}},
            {"version": "region_onehot", **{k: ev_region[k] for k in [
                "subset_accuracy","hamming_loss","f1_micro","f1_macro","roc_auc_macro","pr_auc_macro"
            ]}},
        ])
        comp
    else:
        print("未找到指定的地区列，跳过对比。")
except Exception as e:
    print("对比运行出错：", e)


Epoch 1/1:   0%|          | 0/625 [00:00<?, ?it/s]